# Re-ID 파이프라인 실행 노트북

Google Colab에서 `run_pipeline.sh`를 실행합니다.

| 단계 | 내용 | 파일 |
|------|------|------|
| Stage 1 | YOLO + BotSort → track_id, bbox 수집 | `collect_tracks.ipynb` |
| Stage 2 | tracks.pkl → ReID 임베딩 추출 + 이미지 저장 | `reid_performance.ipynb` |

**끊겨도 재실행하면 자동으로 이어서 처리합니다.**


### 1. Google Drive 마운트


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### 2. 최신 코드 가져오기


In [ ]:
import subprocess, os

REPO_URL   = 'https://github.com/YOUR_USERNAME/EYE-D.git'  # ← 본인 저장소로 수정
REPO_BRANCH = 'feat/phase1'                                 # ← 브랜치 확인
REPO_LOCAL = '/content/drive/MyDrive/projects/EYE-D/EYE-D' # ← DRIVE_ROOT와 동일하게

if os.path.exists(f'{REPO_LOCAL}/.git'):
    print('git pull — 최신 코드 가져오는 중...')
    result = subprocess.run(
        ['git', '-C', REPO_LOCAL, 'pull', 'origin', REPO_BRANCH],
        text=True, capture_output=True
    )
    print(result.stdout or result.stderr)
else:
    print('git clone — 저장소 최초 다운로드 중...')
    parent = os.path.dirname(REPO_LOCAL)
    os.makedirs(parent, exist_ok=True)
    result = subprocess.run(
        ['git', 'clone', '-b', REPO_BRANCH, REPO_URL, REPO_LOCAL],
        text=True, capture_output=True
    )
    print(result.stdout or result.stderr)

if result.returncode != 0:
    raise RuntimeError(f'git 오류:
{result.stderr}')
print('완료')


### 3. 파라미터 설정


In [ ]:
# ── 경로 설정 ─────────────────────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/projects/EYE-D/EYE-D'   # ← 필요 시 수정
DATA_DIR   = f'{DRIVE_ROOT}/data'
SCRIPT     = f'{DRIVE_ROOT}/edge/notebooks/run_pipeline.sh'

# ── 처리할 영상 ────────────────────────────────────────────────────────────────
import glob, os
VIDEO_FILES = sorted(glob.glob(f'{DATA_DIR}/*.avi'))
print(f'처리 대상 영상 {len(VIDEO_FILES)}개:')
for v in VIDEO_FILES:
    print(f'  {os.path.basename(v)}')


In [ ]:
# ── 실행 옵션 ─────────────────────────────────────────────────────────────────
IMAGE_DIR   = 'dataset'   # 학습용 이미지 저장 폴더 ('' = 저장 안 함)
FRAME_STEP  = 5           # Stage 2: N번째 프레임만 ReID 추출
MAX_FRAMES  = 'inf'       # 최대 처리 프레임 (inf = 전체)
CLEAN       = '0'         # '1' = 기존 결과 삭제 후 재실행


### 4. 패키지 설치


In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'papermill', 'ultralytics', 'boxmot'], check=True)
# deep-person-reid (OSNet)
subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/KaiyangZhou/deep-person-reid.git'], check=True)
print('설치 완료')


### 5. 파이프라인 실행


In [ ]:
import subprocess, shlex

if not VIDEO_FILES:
    raise FileNotFoundError(f'영상 파일이 없습니다: {DATA_DIR}/*.avi')

videos_str = ' '.join(shlex.quote(v) for v in VIDEO_FILES)

cmd = (
    f'COLAB=1 '
    f'DRIVE_ROOT={shlex.quote(DRIVE_ROOT)} '
    f'IMAGE_DIR={shlex.quote(IMAGE_DIR)} '
    f'FRAME_STEP={FRAME_STEP} '
    f'MAX_FRAMES={MAX_FRAMES} '
    f'CLEAN={CLEAN} '
    f'bash {shlex.quote(SCRIPT)} {videos_str}'
)

print('실행 명령:')
print(cmd)
print()

result = subprocess.run(cmd, shell=True, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode != 0:
    print(f'[오류] 종료 코드: {result.returncode}')


### 6. 결과 확인


In [ ]:
import os, pickle

RESULTS_DIR = f'{DRIVE_ROOT}/results'
TRACKS_DIR  = f'{DRIVE_ROOT}/tracks'

print('── tracks.pkl ──────────────────────────────────')
for f in sorted(os.listdir(TRACKS_DIR)) if os.path.exists(TRACKS_DIR) else []:
    if not f.endswith('.pkl'):
        continue
    with open(f'{TRACKS_DIR}/{f}', 'rb') as fh:
        d = pickle.load(fh)
    total   = d.get('total_frames', '?')
    done    = d.get('frames_processed', '?')
    n_tracks = len(d.get('tracks', {}))
    print(f'  {f}: {done}/{total} 프레임  |  트랙 {n_tracks}개')

print()
print('── embeddings.pkl ──────────────────────────────')
for f in sorted(os.listdir(RESULTS_DIR)) if os.path.exists(RESULTS_DIR) else []:
    if not f.endswith('.pkl'):
        continue
    with open(f'{RESULTS_DIR}/{f}', 'rb') as fh:
        d = pickle.load(fh)
    done     = d.get('frames_processed', '완료')
    n_tracks = len(d.get('data_x025', {}))
    n_embeds = sum(len(v) for v in d.get('data_x025', {}).values())
    print(f'  {f}: {done} 프레임  |  트랙 {n_tracks}개  |  임베딩 {n_embeds}개')
